In [ ]:
#!pip install --break-system-packages clickhouse-connect

In [1]:
import clickhouse_connect
import pandas as pd
import os

pd.options.display.max_colwidth = 100

# Вбейте свой телеграм никнейм, чтобы в случае проблем мы могли вас индефицировать
database = 'shushiato'

client = clickhouse_connect.get_client(host='clickhouse01', port=8123, username=os.getenv('CLICKHOUSE_USER'), password=os.getenv('CLICKHOUSE_PASSWORD'))



In [ ]:
client.command(f'''
    CREATE DATABASE IF NOT EXISTS {database} ON CLUSTER '{{cluster}}';
''')




In [14]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.type_data;
''')


In [ ]:
# запустить следующий create без комментов

client.command(f'''
    CREATE TABLE {database}.type_data
    (
    --------------------------
    -- основные типы данных --
    --------------------------
        i Int8,                               -- Int8-256 (со знаком)
        ui UInt8,                             -- UInt8-256 (без знака)
        fl Float32,                           -- Float32-64 (для мат.расчетов, но не для финансов)  
        dc Decimal(9, 4),                      -- Decimal32-256 (точность после запятой)
        st String,                            -- имеет произвольную длинну
        fst FixedString(5),                   -- имеет фиксированную длинну
    --------------------------------
    -- дополнитеьлные типы данных --
    --------------------------------
        UID UUID,                             -- уникальный идентификатор
        ip4 IPv4,                             -- 127.0.0.1
        ip6 IPv6,                             -- f2c6:e19b:da60:52ad:2cef:62fe:0279
    --------------------------------
    --    типы даты и времени     --
    --------------------------------
        dt Date,                              -- Date32 (различаются диапозном дат)
        dtm DateTime,                         -- Сохрняет время с точностью то секунд
        dtm64 DateTime64,                     -- Сохрняет время с точностью то наносекунд
    ------------------------------------
    --    композитные типы данных             -- позволяют хранить более сложные структуры данных
    ------------------------------------
        ar Array(UInt8),                      -- массив данных
        tu Tuple(Date, UInt16, Decimal32(2)), -- кортеж
        ns Nested(                            -- Вложенные структуры
            col1 String,
            col2 UInt64,
            col3 String
            ),
    mp Map(String, Int16),                    -- хранит в данные в виде ключ -> значение
    en Enum('bad' = 2,                        -- хранит данные определенного значения
            'udovlet' = 3, 
            'good' = 4 )      
    )
    ENGINE = Log;
''')



In [15]:
# запустить этот create без комментов

client.command(f'''
    CREATE TABLE {database}.type_data
    (
        i Int8,
        ui UInt8,
        fl Float32,
        dc Decimal(9, 4),
        st String,
        fst FixedString(5),
        UID UUID,
        ip4 IPv4,
        ip6 IPv6,
        dt Date,
        dtm DateTime,
        dtm64 DateTime64,
        ar Array(UInt8),
        tu Tuple(Date, UInt16, Decimal32(2)),
        ns Nested(
            col1 String,
            col2 UInt64,
            col3 String
            ),
    mp Map(String, Int16),
    en Enum('bad' = 2,
            'udovlet' = 3, 
            'good' = 4 )      
    )
    ENGINE = Log;
''')


In [16]:
client.command(f'''
    INSERT INTO {database}.type_data
    (
        i, 
        ui, 
        fl, 
        dc, 
        st, 
        fst,
        UID, 
        ip4, 
        ip6,
        dt, 
        dtm, 
        dtm64,
        ar, 
        tu, 
        ns.col1, 
        ns.col2, 
        ns.col3,
        mp, 
        en
    )
    VALUES 
    (
        -100,
        200,
        3.14,
        toDecimal32(3.14, 4),
        'Пример строки',
        'ABCDE',
        generateUUIDv4(),
        '192.168.1.1',
        '2001:db8::1',
        toDate('2025-04-30'),
        toDateTime('2025-04-30 14:30:00'),
        parseDateTime64BestEffort('2025-04-30 14:30:00.123456', 6),
        [10, 20, 30],
        (toDate('2025-04-30'), 150, 99.99),
        ['one'],
        [123456],
        ['value1'],
        {{'key1': 10, 'key2': -20}},
        'udovlet'
    );
''')    

In [ ]:
client.query_df(f'''
    SELECT 
      *
    FROM {database}.type_data
''')



,i,ui,fl,dc,st,fst,UID,ip4,ip6,dt,dtm,dtm64,ar,tu,ns.col1,ns.col2,ns.col3,mp,en
0,-100,200,3.14,3.1400,Пример строки,b'ABCDE',f7c6d308-da02-449e-b264-e71929c04e6c,192.168.1.1,2001:db8::1,2025-04-30,2025-04-30 14:30:00+03:00,2025-04-30 14:30:00.123000+03:00,"[10, 20, 30]","(2025-04-30T00:00:00.000000000, 150, 99.99)",[one],[123456],[value1],"{'key1': 10, 'key2': -20}",udovlet


In [ ]:
#обращение к композитным типам данных
client.query_df(f'''
    SELECT 
        ar[1],      -- обращение к эл-там массива
        ar.size0,   -- получение размера массива
        tu,         -- чтение кортежа
        ns.col1,    -- обращение к вложенной структуре
        mp['key1'], -- получение данных из Map
        en          -- Чтение Enum
    FROM {database}.type_data
''')


,"arrayElement(ar, 1)",ar.size0,tu,ns.col1,"arrayElement(mp, 'key1')",en
0,10,3,"(2025-04-30T00:00:00.000000000, 150, 99.99)",[one],10,udovlet


1) Необходимо содать таблицу заказов (Мы еще не проходили создание таблиц поэтому создай по примеру выше с движком `Log`)

| Поле         | Описание                            |
|--------------|-------------------------------------|
| `order_id`   | Уникальный ID заказа                |
| `user_id`    | ID пользователя                     |
| `order_date` | Дата и время оформления заказа      |
| `total_amount` | Сумма заказа                      |
| `paid`       |  Признак оплаты: оплачено, не оплачено|
| `items`      |  Список ID товаров в заказе         |

2) Тебе необходимо выбрать самому подходящие типы данных для колонок
3) После создания вставь подходящие строки через `INSERT INTO <имя БД>.<имя таблицы> VALUES (...)`
4) Выведи выборку строк

In [ ]:
#создание таблицы заказов
client.command(f'''
    CREATE TABLE IF NOT EXISTS {database}.orders (
        order_id UInt64,                    -- Уникальный ID заказа
        user_id UInt32,                     -- ID пользователя  
        order_date DateTime64(3),           -- Дата/время с миллисекундами
        total_amount Decimal(10, 2),        -- Сумма с 2 знаками после запятой
        paid Enum8('unpaid' = 0, 'paid' = 1),   -- Признак оплаты
        items Array(UInt32)                 -- Список ID товаров
    ) ENGINE = Log;
''')


In [ ]:

#вставка данных
client.command(f'''
    INSERT INTO {database}.orders VALUES 
    (1,  1001, '2026-02-02 10:30:15.123',  1250.50, 'paid', [101, 102, 105]),
    (2,  1002, '2026-02-02 11:15:22.456',   890.75, 'unpaid',  [103, 104]),
    (3,  1001, '2026-02-02 14:20:08.789',  2345.00, 'paid', [102, 106, 107, 108]),
    (4,  1003, '2026-02-02 16:45:33.000',   450.25, 'paid', [101]),
    (5,  1004, '2026-02-02 18:12:47.999',  1789.99, 'unpaid',  [105, 103, 109]);
''')



In [ ]:
#проверка что данные вставлены
client.query_df('''
    SELECT 
        order_id,
        user_id, 
        order_date,
        total_amount,
        paid,
        items,
        arrayJoin(items) AS item_id,
        length(items) AS items_count
    FROM shushiato.orders 
    ORDER BY order_date 
    LIMIT 20
''')



,order_id,user_id,order_date,total_amount,paid,items,item_id,items_count
0,1,1001,2026-02-02 10:30:15.123000+03:00,1250.50,paid,"[101, 102, 105]",101,3
1,1,1001,2026-02-02 10:30:15.123000+03:00,1250.50,paid,"[101, 102, 105]",102,3
2,1,1001,2026-02-02 10:30:15.123000+03:00,1250.50,paid,"[101, 102, 105]",105,3
3,2,1002,2026-02-02 11:15:22.456000+03:00,890.75,unpaid,"[103, 104]",103,2
4,2,1002,2026-02-02 11:15:22.456000+03:00,890.75,unpaid,"[103, 104]",104,2
5,3,1001,2026-02-02 14:20:08.789000+03:00,2345.00,paid,"[102, 106, 107, 108]",102,4
6,3,1001,2026-02-02 14:20:08.789000+03:00,2345.00,paid,"[102, 106, 107, 108]",106,4
7,3,1001,2026-02-02 14:20:08.789000+03:00,2345.00,paid,"[102, 106, 107, 108]",107,4
8,3,1001,2026-02-02 14:20:08.789000+03:00,2345.00,paid,"[102, 106, 107, 108]",108,4
9,4,1003,2026-02-02 16:45:33+03:00,450.25,paid,[101],101,1


In [ ]:
# **Функции к приведению типов данных**
client.query_df('''
    select '1'::Int64
''')
client.query_df('''
    select CAST('1'  as Int8)
''')

,"CAST('1', 'Int64')"
0,1
